In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Source Han Sans CN']  # 优先使用系统已有的中文字体

In [5]:
# 1.1 导入数据
print("=" * 50)
print("1.1 数据导入")
print("=" * 50)

df = pd.read_csv('rent_price_final_train_dataset.csv')

print(f"数据形状: {df.shape}")
print(f"行数: {df.shape[0]}, 列数: {df.shape[1]}")
print("\n数据基本信息:")
print(df.info())

1.1 数据导入
数据形状: (98899, 57)
行数: 98899, 列数: 57

数据基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 57 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Price                   98899 non-null  float64
 1   lnPrice                 98899 non-null  float64
 2   decoration              98899 non-null  int64  
 3   high_dummy              98894 non-null  float64
 4   middle_dummy            98894 non-null  float64
 5   low_dummy               98894 non-null  float64
 6   basement_dummy          98894 non-null  float64
 7   total_floor             98894 non-null  float64
 8   area                    98899 non-null  float64
 9   room_count              98865 non-null  float64
 10  hall_count              93871 non-null  float64
 11  south_dummy             98894 non-null  float64
 12  north_south_dummy       98894 non-null  float64
 13  pay_annual              80471 non-nul

In [6]:
# 1.2 缺失值分析
print("=" * 50)
print("1.2 缺失值分析")
print("=" * 50)

# 计算所有列的缺失值比例
missing_analysis = pd.DataFrame({
    '缺失数量': df.isnull().sum(),
    '缺失比例(%)': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('缺失比例(%)', ascending=False)

print("所有变量的缺失值统计:")
print(missing_analysis)

# 输出缺失值较多的变量
high_missing = missing_analysis[missing_analysis['缺失比例(%)'] > 50]
if not high_missing.empty:
    print(f"\n缺失比例超过50%的变量 ({len(high_missing)}个):")
    for var in high_missing.index:
        print(f"  {var}: {high_missing.loc[var, '缺失比例(%)']}%")

1.2 缺失值分析
所有变量的缺失值统计:
                         缺失数量  缺失比例(%)
heating_fee_avg         70064    70.84
heating_self            59664    60.33
lease_max_months        46933    47.46
lease_min_months        46933    47.46
lease_avg_months        46933    47.46
facility_空调             30451    30.79
facility_衣柜             30451    30.79
facility_洗衣机            30451    30.79
facility_冰箱             30451    30.79
facility_天然气            30451    30.79
facility_电视             30451    30.79
facility_宽带             30451    30.79
facility_暖气             30451    30.79
facility_热水器            30451    30.79
facility_床              30451    30.79
building_age            26149    26.44
parking_spots           25479    25.76
gas_fee_avg             25057    25.34
greening_rate           24402    24.67
plot_ratio              24080    24.35
property_fee_avg        22159    22.41
pay_annual              18428    18.63
pay_bi_monthly          18428    18.63
pay_monthly             18428    18.63
pay

In [7]:
# 1.3 连续型变量填充
print("\n" + "=" * 50)
print("1.3 连续型变量填充")
print("=" * 50)

# 根据你提供的缺失变量列表，定义需要填充的连续型变量
continuous_variables_to_fill = [
    'heating_fee_avg', 'lease_avg_months', 'lease_min_months', 'lease_max_months',
    'building_age', 'parking_spots', 'gas_fee_avg', 'greening_rate', 
    'plot_ratio', 'property_fee_avg', 'hall_count', 'household_total', 
    'building_total', 'room_count', 'total_floor'
]

print("开始填充连续型变量...")
for column in continuous_variables_to_fill:
    if column in df.columns:
        null_count_before = df[column].isnull().sum()
        if null_count_before > 0:
            # 第一层：按板块分组计算中位数并填充
            df[column] = df.groupby('板块')[column].transform(
                lambda x: x.fillna(x.median())
            )
            
            # 第二层：检查是否仍有缺失值，如果有则按区县分组填充
            null_count_after_plate = df[column].isnull().sum()
            if null_count_after_plate > 0:
                df[column] = df.groupby('区县')[column].transform(
                    lambda x: x.fillna(x.median())
                )
            
            # 第三层：检查是否仍有缺失值，如果有则按城市分组填充
            null_count_after_district = df[column].isnull().sum()
            if null_count_after_district > 0:
                df[column] = df.groupby('城市')[column].transform(
                    lambda x: x.fillna(x.median())
                )
            
            # 第四层：检查是否仍有缺失值，如果有则用全局中位数填充
            null_count_after_city = df[column].isnull().sum()
            if null_count_after_city > 0:
                global_median = df[column].median()
                df[column] = df[column].fillna(global_median)
            
            null_count_after = df[column].isnull().sum()
            print(f"{column}: {null_count_before} → {null_count_after} 缺失值")
        else:
            print(f"{column}: 无缺失值")
    else:
        print(f"{column}: 列不存在")

print("\n✅ 连续型变量填充完成")


1.3 连续型变量填充
开始填充连续型变量...
heating_fee_avg: 70064 → 0 缺失值
lease_avg_months: 46933 → 0 缺失值
lease_min_months: 46933 → 0 缺失值
lease_max_months: 46933 → 0 缺失值
building_age: 26149 → 0 缺失值
parking_spots: 25479 → 0 缺失值
gas_fee_avg: 25057 → 0 缺失值
greening_rate: 24402 → 0 缺失值
plot_ratio: 24080 → 0 缺失值
property_fee_avg: 22159 → 0 缺失值
hall_count: 5028 → 0 缺失值
household_total: 4631 → 0 缺失值
building_total: 4631 → 0 缺失值
room_count: 34 → 0 缺失值
total_floor: 5 → 0 缺失值

✅ 连续型变量填充完成


In [9]:
# 1.4 分类变量和哑变量填充
print("\n" + "=" * 50)
print("1.4 分类变量和哑变量填充")
print("=" * 50)

# 定义需要填充的分类变量和哑变量
categorical_vars_to_fill = [
    'heating_self', 'facility_洗衣机', 'facility_电视', 'facility_空调', 
    'facility_衣柜', 'facility_宽带', 'facility_暖气', 'facility_热水器', 
    'facility_冰箱', 'facility_天然气', 'facility_床', 'pay_bi_monthly', 
    'pay_quarterly', 'pay_semi_annual', 'pay_monthly', 'pay_annual',
    'water_civil', 'water_commercial', 'electricity_commercial', 
    'electricity_civil', 'gas_yes', 'low_dummy', 'middle_dummy', 
    'high_dummy', 'basement_dummy', 'south_dummy', 'north_south_dummy',
    'elevator_yes'
]

print("开始填充分类变量和哑变量...")
for column in categorical_vars_to_fill:
    if column in df.columns:
        null_count_before = df[column].isnull().sum()
        if null_count_before > 0:
            # 第一层：按板块分组用众数填充
            df[column] = df.groupby('板块')[column].transform(
                lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
            )
            
            # 第二层：检查是否仍有缺失值，如果有则按区县分组用众数填充
            null_count_after_plate = df[column].isnull().sum()
            if null_count_after_plate > 0:
                df[column] = df.groupby('区县')[column].transform(
                    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
                )
            
            # 第三层：检查是否仍有缺失值，如果有则按城市分组用众数填充
            null_count_after_district = df[column].isnull().sum()
            if null_count_after_district > 0:
                df[column] = df.groupby('城市')[column].transform(
                    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
                )
            
            # 第四层：检查是否仍有缺失值，如果有则用全局众数填充
            null_count_after_city = df[column].isnull().sum()
            if null_count_after_city > 0:
                global_mode = df[column].mode()[0] if not df[column].mode().empty else 0
                df[column] = df[column].fillna(global_mode)
            
            null_count_after = df[column].isnull().sum()
            print(f"{column}: {null_count_before} → {null_count_after} 缺失值")
        else:
            print(f"{column}: 无缺失值")
    else:
        print(f"{column}: 列不存在")

print("\n✅ 分类变量和哑变量填充完成")


1.4 分类变量和哑变量填充
开始填充分类变量和哑变量...
heating_self: 59664 → 0 缺失值
facility_洗衣机: 30451 → 0 缺失值
facility_电视: 30451 → 0 缺失值
facility_空调: 30451 → 0 缺失值
facility_衣柜: 30451 → 0 缺失值
facility_宽带: 30451 → 0 缺失值
facility_暖气: 30451 → 0 缺失值
facility_热水器: 30451 → 0 缺失值
facility_冰箱: 30451 → 0 缺失值
facility_天然气: 30451 → 0 缺失值
facility_床: 30451 → 0 缺失值
pay_bi_monthly: 18428 → 0 缺失值
pay_quarterly: 18428 → 0 缺失值
pay_semi_annual: 18428 → 0 缺失值
pay_monthly: 18428 → 0 缺失值
pay_annual: 18428 → 0 缺失值
water_civil: 8696 → 0 缺失值
water_commercial: 8696 → 0 缺失值
electricity_commercial: 8489 → 0 缺失值
electricity_civil: 8489 → 0 缺失值
gas_yes: 2880 → 0 缺失值
low_dummy: 5 → 0 缺失值
middle_dummy: 5 → 0 缺失值
high_dummy: 5 → 0 缺失值
basement_dummy: 5 → 0 缺失值
south_dummy: 5 → 0 缺失值
north_south_dummy: 5 → 0 缺失值
elevator_yes: 4 → 0 缺失值

✅ 分类变量和哑变量填充完成


In [10]:
# 1.5 验证填充结果
print("\n" + "=" * 50)
print("1.5 填充结果验证")
print("=" * 50)

# 检查填充后的缺失值情况
missing_after = df.isnull().sum()
missing_after_percent = (missing_after / len(df)) * 100
missing_after_df = pd.DataFrame({
    '缺失数量': missing_after,
    '缺失比例': missing_after_percent
}).sort_values('缺失比例', ascending=False)

# 只显示仍有缺失值的列
missing_after_df = missing_after_df[missing_after_df['缺失数量'] > 0]

if len(missing_after_df) == 0:
    print("✅ 所有变量的缺失值都已成功填充！")
else:
    print("⚠️ 以下变量仍有缺失值:")
    print(missing_after_df)

# 1.6 保存处理后的数据
print("\n" + "=" * 50)
print("1.6 保存数据")
print("=" * 50)

output_file = 'rent_price_final_train_dataset2.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"✅ 处理后的数据已保存为: {output_file}")
print(f"最终数据形状: {df.shape}")

# 显示数据基本信息
print("\n最终数据基本信息:")
print(df.info())


1.5 填充结果验证
⚠️ 以下变量仍有缺失值:
    缺失数量      缺失比例
板块  5144  5.201266
区县  4677  4.729067

1.6 保存数据
✅ 处理后的数据已保存为: rent_price_final_train_dataset2.csv
最终数据形状: (98899, 57)

最终数据基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 57 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Price                   98899 non-null  float64
 1   lnPrice                 98899 non-null  float64
 2   decoration              98899 non-null  int64  
 3   high_dummy              98899 non-null  float64
 4   middle_dummy            98899 non-null  float64
 5   low_dummy               98899 non-null  float64
 6   basement_dummy          98899 non-null  float64
 7   total_floor             98899 non-null  float64
 8   area                    98899 non-null  float64
 9   room_count              98899 non-null  float64
 10  hall_count              98899 non-null  float64
 11  south_dummy            